# 02 Lógica Híbrida de Cruzamento (Spatial Join + Fuzzy String)

**Objetivo:** Cruzar a base do CNEFE 2022 (IBGE) com o Gold Standard (BHMap Endereços) para parear pontos que representam o mesmo endereço no mundo físico. A avaliação de incerteza modelará o `MCI` (Matching Certainty Indicator).

**Entradas (GeoParquet):**
- `data/processed/cnefe_bh.parquet`
- `data/processed/bhmap_enderecos.parquet`

**Saídas:**
- `data/interim/matched_addresses.parquet`
- `outputs/tables/02_match_profile.csv`

**Dependências:** geopandas, pandas, rapidfuzz

**Hardware/Tempo Esperado:** Cerca de 1 a 3 minutos para 800k registros (o cálculo fuzzy roda em C++ via rapidfuzz).

In [1]:
import sys
from pathlib import Path
import os

# Ensure root path is accessible to import src
os.chdir('..')
sys.path.append(os.getcwd())

%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm
tqdm.pandas()

from src import config
from src.logging import logger
from src.matching import get_spatial_candidates, calculate_mci, resolve_best_match

## 1. Carregamento dos Dados Processados

In [2]:
logger.info("Carregando bases GeoParquet do UTM 23S")
gdf_cnefe_bh = gpd.read_parquet(config.CNEFE_PROCESSED_FILE)
gdf_bhmap = gpd.read_parquet(config.BHMAP_PROCESSED_FILE)

logger.info("Amostra dos schemas para certificar que STD_ está presente")
display(gdf_cnefe_bh[['std_logradouro_completo']].head(2))
display(gdf_bhmap[['std_tipo_logradouro', 'std_nome_logradouro', 'std_numero']].head(2))

2026-03-03T02:26:28.494742Z [info     ] Carregando bases GeoParquet do UTM 23S [__main__]


2026-03-03T02:26:31.219173Z [info     ] Amostra dos schemas para certificar que STD_ está presente [__main__]


,std_logradouro_completo
0,"RUA DIALOGITA, 135"
1,"AVENIDA DOM PEDRO II, 3687"


,std_tipo_logradouro,std_nome_logradouro,std_numero
0,BEC,SERVIDAO,
1,AVE,PERIMETRAL,


## 2. Busca Espacial de Candidatos (R-Tree: Raio de 50 Metros)
Aqui nós superamos a limitação do *String Matching* puro. Em vez de comparar as strings do CNEFE contra todos os milhões do BHMap (fórmula O(N²)), nós restringimos os candidatos de cada ponto apenas aos endereços da prefeitura localizados fisicamente a no máximo **50 metros** do agente censitário. 

> **Teoria (Davis & Fonseca, 2007)**: Reduz-se drasticamente os falsos positivos de endereços homônimos isolando os vizinhos.

In [3]:
logger.info("Inicializando motor R-Tree via pygeos/rtree")
df_candidates = get_spatial_candidates(gdf_cnefe_bh, gdf_bhmap, max_distance=50.0)

logger.info("Estatísticas dos vizinhos obtidos:")
display(df_candidates[['id_cnefe', 'id_bhmap', 'spatial_distance']].head(5))

2026-03-03T02:26:31.344661Z [info     ] Inicializando motor R-Tree via pygeos/rtree [__main__]


2026-03-03T02:26:31.344661Z [info     ] Starting Spatial Join (sjoin_nearest) [__main__] max_distance=50.0


2026-03-03T02:26:49.334958Z [info     ] Spatial Join completed         [__main__] total_candidates=1185811


2026-03-03T02:26:49.426209Z [info     ] Estatísticas dos vizinhos obtidos: [__main__]


,id_cnefe,id_bhmap,spatial_distance
0,0,361897.0,7.805313
1,1,143365.0,3.983400
2,2,303950.0,7.045431
3,3,126869.0,3.329680
4,4,182309.0,2.320929


## 3. NLP Fuzzy Logic & MCI (Matching Certainty Indicator)
Agora, para cada candidato vizinho (às vezes 5, 10 endereços do BHMap num raio de 50 metros), rodamos o avaliador da distância de *Levenshtein* através da função de ordenamento de tokens de alto desempenho `fuzz.token_sort_ratio`.

In [4]:
logger.info("Aplicando avaliador de Incerteza (MCI) na matriz de O(N)...")

# O MCI varia de 0.0 (totalmente diferente foneticamente) a 1.0 (match semântico perfeito na mesma calçada)
# NOTA: Dependendo do número de candidatos, esta etapa pode demorar alguns minutos.
# O `apply` interno da função usa Pandas otimizado. ProgressBar integrada não reflete `apply` de série, apenas o log final.

df_candidates = calculate_mci(df_candidates)

display(df_candidates[['std_logradouro_completo', 'std_nome_logradouro', 'spatial_distance', 'MCI']].sample(5))

2026-03-03T02:26:49.491369Z [info     ] Aplicando avaliador de Incerteza (MCI) na matriz de O(N)... [__main__]


2026-03-03T02:26:49.492761Z [info     ] Calculating Fuzzy String Matching (MCI) [__main__]


2026-03-03T02:26:53.881977Z [info     ] Fuzzy logic (MCI) calculation completed [__main__]


,std_logradouro_completo,std_nome_logradouro,spatial_distance,MCI
901220,"RUA DAS PETUNIAS, 3109 SOBRADO",DAS PETUNIAS,4.323370,0.711111
142116,"RUA DOUTOR JULIO OTAVIANO FERREIRA, 472",DOUTOR JULIO OTAVIANO FERREIRA,10.770457,0.944444
450524,"RUA SANTOS, 689",SANTOS,7.319672,0.833333
294942,"RUA MANHUMIRIM, 925",MANHUMIRIM,8.359510,0.875000
1092798,"AVENIDA AFONSO PENA, 367",AFONSO PENA,5.500545,0.789474


## 4. Resolução Desambiguadora Final
Mantém apenas o melhor candidato para cada ponto isolado do CNEFE (maior similaridade textual MCI). Se houver empate gramatical no mesmo quarteirão (0.8 e 0.8), pegamos o de menor distância euclidiana.

In [5]:
gdf_matched = resolve_best_match(df_candidates)

# Count final statistics
total_points = len(gdf_cnefe_bh)
perfect_matches = len(gdf_matched[gdf_matched['MCI'] == 1.0])
good_matches = len(gdf_matched[(gdf_matched['MCI'] >= 0.75) & (gdf_matched['MCI'] < 1.0)])
weak_matches = len(gdf_matched[(gdf_matched['MCI'] > 0) & (gdf_matched['MCI'] < 0.75)])
orphans = len(gdf_matched[gdf_matched['MCI'] == 0.0])

profile_stats = pd.DataFrame([{
    'Perfect Matches (MCI = 1.0)': perfect_matches,
    'High Certainty (0.75 <= MCI < 1.0)': good_matches,
    'Low Certainty (0.0 < MCI < 0.75)': weak_matches,
    'Lost / Unmatched (Orphans > 50m)': orphans,
    'Total Originais CNEFE': total_points
}])

display(profile_stats)

2026-03-03T02:26:54.127911Z [info     ] Resolving best matches from candidates [__main__]


2026-03-03T02:26:55.531805Z [info     ] Best match resolution completed [__main__] final_rows=1180102


,Perfect Matches (MCI = 1.0),High Certainty (0.75 <= MCI < 1.0),Low Certainty (0.0 < MCI < 0.75),Lost / Unmatched (Orphans > 50m),Total Originais CNEFE
0,0,987820,174766,17516,1180102


## 5. Exportação da Base Avaliada

In [6]:
MATCHED_FILE = config.INTERIM_DATA_DIR / "cnefe_match_bhmap.parquet"
gdf_matched.to_parquet(MATCHED_FILE)
logger.info("Matriz de Similaridades consolidada em disco.", arquivo=str(MATCHED_FILE))

STATS_FILE = config.TABLES_DIR / "02_match_profile.csv"
profile_stats.to_csv(STATS_FILE, index=False)
logger.info("Tabela CSV de Match Profiling exportada.", arquivo=str(STATS_FILE))

2026-03-03T02:27:00.612504Z [info     ] Matriz de Similaridades consolidada em disco. [__main__] arquivo=C:\Users\mateu\OneDrive\Documentos\UFMG\Mestrado\geocoding-quality-analysis\data\interim\cnefe_match_bhmap.parquet


2026-03-03T02:27:00.619319Z [info     ] Tabela CSV de Match Profiling exportada. [__main__] arquivo=C:\Users\mateu\OneDrive\Documentos\UFMG\Mestrado\geocoding-quality-analysis\outputs\tables\02_match_profile.csv
